# Домашняя работа 3

## Подготовка

In [1]:
from pathlib import Path
import pandas as pd
from pandas.testing import assert_frame_equal
DATA = Path("../data")
WORK = Path("../results")
WORK.mkdir(exist_ok=True)
decisions = pd.read_csv(
    DATA / "decisions.csv", dtype={"lei": "string", "county_code": "string"}
)

results_dir = Path("../results")
results_dir.mkdir(parents=True, exist_ok=True)
print("pandas", pd.__version__, decisions.shape)

pandas 2.2.2 (6830, 7)


## Исходники домашней работы

In [2]:
org_keys = ["lei", "activity_year"]
long_keys = org_keys + ["action_taken"]
organizations = pd.read_csv(DATA / "organizations.csv", dtype={"lei": "string"})
lookup = organizations[org_keys + ["organization_name"]].copy()
assert not lookup.duplicated(org_keys).any()
print(organizations.head(3))

                    lei                       organization_name  source_count  \
0  549300AG64NHILB7ZP05                      LOANDEPOT.COM, LLC           114   
1  QFROUN1UWUYU0DVIWD51  Fifth Third Bank, National Association             3   
2  5493003GQDUH26DNNH17               NAVY FEDERAL CREDIT UNION           742   

   activity_year  
0           2023  
1           2023  
2           2023  


## H1 Профиль отказов

In [3]:
dec_h1 = decisions.copy()
dec_h1["is_denied"] = dec_h1["action_taken"].eq(3)

denial = (
    dec_h1.groupby(org_keys, as_index=False)
    .agg(
        n_decisions=("is_denied", "count"),
        n_denied=("is_denied", "sum"),
    )
)
denial["denial_rate"] = denial["n_denied"] / denial["n_decisions"]

denial_merged = denial.merge(
    lookup,
    on=org_keys,
    how="left",
    validate="one_to_one",
    indicator=True,
)

print("Покрытие JOIN:", denial_merged["_merge"].value_counts().to_dict())
assert (denial_merged["_merge"] == "both").all()
assert denial_merged["n_decisions"].sum() == len(decisions)
assert denial_merged["n_denied"].sum() == dec_h1["is_denied"].sum()
assert denial_merged["denial_rate"].between(0, 1).all()

denial = denial_merged.drop(columns=["_merge"])
denial.to_csv(results_dir / "home_denial_profile.csv", index=False)

display(denial.head())

Покрытие JOIN: {'both': 314, 'left_only': 0, 'right_only': 0}


,lei,activity_year,n_decisions,n_denied,denial_rate,organization_name
0,03D0JEWFDFUS0SEEKG89,2023,91,15,0.164835,"TD Bank, National Association"
1,1IE8VN30JCEQV1H4R804,2023,7,2,0.285714,"HSBC BANK USA, NATIONAL ASSOCIATION"
2,213800KVD76WKI6PN868,2023,2,1,0.500000,"CAPITAL CENTER, L.L.C."
3,213800QUAI2VH5YM6310,2023,89,9,0.101124,"EMBRACE HOME LOANS, INC."
4,213800XR2TCBQJSF1X93,2023,15,3,0.200000,NBKC BANK


## H2 Группы для просмотра

In [4]:
large_groups = denial[denial["n_decisions"].ge(20)].copy()
large_groups.to_csv(results_dir / "home_denial_groups20.csv", index=False)

overall_denial_rate = denial["n_denied"].sum() / denial["n_decisions"].sum()
simple_mean_rate = denial["denial_rate"].mean()

print(f"Всего групп (организаций): {len(denial)}")
print(f"Осталось групп с >= 20 решениями: {len(large_groups)}")
print(f"Общая доля отказов (по суммарным счётчикам): {overall_denial_rate:.4%}")
print(f"Простое среднее долей по организациям: {simple_mean_rate:.4%}")

Всего групп (организаций): 314
Осталось групп с >= 20 решениями: 61
Общая доля отказов (по суммарным счётчикам): 10.1171%
Простое среднее долей по организациям: 14.2224%


## H3 Длинная, широкая и снова длинная форма

In [5]:
counts_initial = (
    decisions.groupby(org_keys + ["action_taken"], as_index=False)
    .size()
    .rename(columns={"size": "n"})
)

wide = (
    counts_initial.pivot(index=org_keys, columns="action_taken", values="n")
    .reindex(columns=[1, 2, 3])
    .fillna(0)
)

long_restored = (
    wide.reset_index()
    .melt(
        id_vars=org_keys,
        value_vars=[1, 2, 3],
        var_name="action_taken",
        value_name="n",
    )
)
long_restored["action_taken"] = long_restored["action_taken"].astype("int64")
long_restored["n"] = long_restored["n"].astype("int64")
long_restored = long_restored[long_restored["n"] > 0].copy()


sort_keys = org_keys + ["action_taken"]
df_left = counts_initial.sort_values(sort_keys).reset_index(drop=True)
df_right = long_restored[counts_initial.columns].sort_values(sort_keys).reset_index(drop=True)

pd.testing.assert_frame_equal(df_left, df_right)
print("Проверка assert_frame_equal пройдена успешно!")

Проверка assert_frame_equal пройдена успешно!


## H+ Безопасное соединение